### Setting Up Kaggle Dataset for Sentiment Analysis

This notebook automates the process of downloading the Amazon Product Reviews Dataset from Kaggle, extracting category-specific files, and uploading them to Google Cloud Storage (GCS). The dataset will be used for sentiment analysis of Amazon product reviews

In [ ]:
# Install Kaggle API
!pip install kaggle

# Create directory for Kaggle credentials
!mkdir -p ~/.kaggle

# Copy the kaggle.json to the right location
!cp kaggle.json ~/.kaggle/

# Set proper permissions
!chmod 600 ~/.kaggle/kaggle.json

# Verify installation
!kaggle datasets list

In [2]:
# Create a temporary directory for the download
!mkdir -p /tmp/amazon_data

# Download the amazon dataset
!kaggle datasets download cynthiarempel/amazon-us-customer-reviews-dataset -p /tmp/amazon_data

# List the downloaded files to verify
!ls -la /tmp/amazon_data

Dataset URL: https://www.kaggle.com/datasets/cynthiarempel/amazon-us-customer-reviews-dataset
License(s): other
100%|██████████████████████████████████████▉| 21.0G/21.0G [03:08<00:00, 151MB/s]
100%|███████████████████████████████████████| 21.0G/21.0G [03:08<00:00, 119MB/s]
total 21970456
drwxr-xr-x  2 root root        4096 Mar  7 14:22 .
drwxrwxrwt 36 root root        4096 Mar  7 14:22 ..
-rw-r--r--  1 root root 22497731749 Jun 16  2021 amazon-us-customer-reviews-dataset.zip


In [3]:
# Copy the zip file to the bucket
!gsutil cp /tmp/amazon_data/amazon-us-customer-reviews-dataset.zip gs://final-project-bucket-amazon/

Copying file:///tmp/amazon_data/amazon-us-customer-reviews-dataset.zip [Content-Type=application/zip]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

- [1 files][ 21.0 GiB/ 21.0 GiB]   81.6 MiB/s                                   
Operation completed over 1 objects/21.0 GiB.                                     


We are selecting these six category files (Apparel, Beauty, Books, Electronics, Furniture, and Mobile Electronics) to provide a diverse representation of product types that cover both physical goods and digital products, offering a balanced dataset for our sentiment analysis across different consumer experiences

In [11]:
# Create directories
!mkdir -p /tmp/extracted

# Updated category extraction with correct file patterns
categories = [
    {"name": "Books", "pattern": "*Books*"},
    {"name": "Electronics", "pattern": "*Electronics*"},
    {"name": "Apparel", "pattern": "*Apparel*"},
    {"name": "Furniture", "pattern": "*Furniture*"},  
    {"name": "Beauty", "pattern": "*Beauty*"}
]

for category in categories:
    print(f"Extracting {category['name']} reviews...")
    !unzip -j /tmp/amazon_data/amazon-us-customer-reviews-dataset.zip "{category['pattern']}" -d /tmp/extracted
    
    # Check if files were extracted
    file_count = !ls -1 /tmp/extracted | wc -l
    if int(file_count[0]) > 0:
        !gsutil -m cp /tmp/extracted/*.tsv gs://final-project-bucket-amazon/amazon-reviews/
        !rm /tmp/extracted/*
        print(f"Successfully extracted and uploaded {category['name']} files")
    else:
        print(f"No files matched pattern {category['pattern']}")

Extracting Books reviews...
Archive:  /tmp/amazon_data/amazon-us-customer-reviews-dataset.zip
  inflating: /tmp/extracted/amazon_reviews_us_Books_v1_02.tsv  
Copying file:///tmp/extracted/amazon_reviews_us_Books_v1_02.tsv [Content-Type=text/tab-separated-values]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

/ [1/1 files][  3.0 GiB/  3.0 GiB] 100% Done  75

In [12]:
!gsutil ls gs://final-project-bucket-amazon/amazon-reviews/

gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Apparel_v1_00.tsv
gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Beauty_v1_00.tsv
gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Books_v1_02.tsv
gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Electronics_v1_00.tsv
gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Furniture_v1_00.tsv
gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Mobile_Electronics_v1_00.tsv
